# 第 3 章:注意力机制(最关键的一章)

**核心问题**:模型如何让每个 token 参考序列里其他 token 的信息?

**章节内容**(循序渐进 4 步)
1. 简化版注意力 —— 不带可学习权重,只做点积+加权求和,先建立直觉
2. 标准 self-attention —— 加入 W_Q, W_K, W_V 三个投影
3. Causal attention —— 加因果掩码,防止看到未来 token
4. Multi-head attention —— 并行多个头捕捉不同模式

**产出**:`llm/attention.py`

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

---
## 1. 简化版注意力(建立直觉)

先不加任何可学习参数,只用最原始的思路:

> 一个 token 的新表示 = 所有 token 的加权平均,权重由「它和其他 token 的相似度」决定。

**公式**
```
scores[i,j] = x[i] · x[j]          # 点积衡量相似度
weights     = softmax(scores, -1)  # 归一化到 [0,1] 且和为 1
output[i]   = sum_j weights[i,j] * x[j]
```

In [ ]:
# 6 个 token,每个 3 维(方便手工查看)
x = torch.tensor([
    [0.43, 0.15, 0.89],  # your
    [0.55, 0.87, 0.66],  # journey
    [0.57, 0.85, 0.64],  # starts
    [0.22, 0.58, 0.33],  # with
    [0.77, 0.25, 0.10],  # one
    [0.05, 0.80, 0.55],  # step
])
print('x shape:', x.shape)

# TODO: 计算 attention scores = x @ x.T
scores = ...
print('scores shape:', scores.shape)  # (6, 6)

# TODO: 对最后一维做 softmax 得到权重
weights = ...
print('weights[0] (每个 token 对第 0 个的关注度):', weights[0])
print('weights[0] 之和 =', weights[0].sum().item())  # 应为 1.0

# TODO: 用权重对 x 加权求和得到输出
context = ...
print('output shape:', context.shape)  # (6, 3)

---
## 2. 标准 Self-Attention(加可学习权重)

上一节里,同一个 x 既当了「查询」又当了「被查的键和值」。这不够灵活,
标准做法是分出三个角色,各配一个可学习投影矩阵:

```
Q = X @ W_Q   # Query:  我在找什么
K = X @ W_K   # Key:    我能被什么查询
V = X @ W_V   # Value:  我实际的内容

attention(Q, K, V) = softmax(Q @ K.T / sqrt(d_k)) @ V
```

**为什么除以 sqrt(d_k)?**
d_k 越大,点积的方差越大 → softmax 容易饱和(梯度接近 0)。
除以 sqrt(d_k) 把方差拉回 1,让训练稳定。

In [ ]:
# TODO: 实现 SelfAttention

class SelfAttention(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        # TODO: 定义三个 nn.Linear(d_in, d_out, bias=False)
        # self.W_Q = ...
        # self.W_K = ...
        # self.W_V = ...
        ...

    def forward(self, x):
        # x: (batch, seq_len, d_in)
        # TODO:
        # 1) Q = self.W_Q(x)   K = self.W_K(x)   V = self.W_V(x)
        # 2) scores  = Q @ K.transpose(-2, -1)              # (B, T, T)
        # 3) scores  = scores / (K.shape[-1] ** 0.5)
        # 4) weights = F.softmax(scores, dim=-1)
        # 5) return weights @ V
        ...


# 验证
xb = x.unsqueeze(0)  # 加 batch 维 -> (1, 6, 3)
attn = SelfAttention(d_in=3, d_out=4)
out = attn(xb)
print('输入 shape:', xb.shape)
print('输出 shape:', out.shape)  # (1, 6, 4)

---
## 3. Causal Attention(加因果掩码)

GPT 是自回归模型:预测第 t 个 token 时,**只能看到位置 0..t-1**,不能偷看未来。

**做法**:在 softmax 之前,把「未来位置」的 score 设为 `-inf`。
softmax 之后这些位置的权重就变成 0。

```
mask = 上三角矩阵(对角线以上为 True)
scores[mask] = -inf
weights = softmax(scores)   # 未来位置权重 = 0
```

**为什么用 -inf 而不是 0?**
softmax 里 `exp(-inf) = 0`,但 `exp(0) = 1`。直接把 score 设 0 不会让权重变 0。

In [ ]:
# 演示 mask
seq_len = 6
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
print('上三角 mask (True 表示要遮挡):')
print(mask)

In [ ]:
# TODO: 实现 CausalAttention

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout=0.0):
        super().__init__()
        self.d_out = d_out
        self.W_Q = nn.Linear(d_in, d_out, bias=False)
        self.W_K = nn.Linear(d_in, d_out, bias=False)
        self.W_V = nn.Linear(d_in, d_out, bias=False)
        self.dropout = nn.Dropout(dropout)
        # register_buffer 把 mask 随模型保存/转设备,但不当作可训练参数
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1).bool()
        )

    def forward(self, x):
        B, T, _ = x.shape
        Q = self.W_Q(x)
        K = self.W_K(x)
        V = self.W_V(x)

        # TODO:
        # 1) scores = Q @ K.transpose(-2, -1)
        # 2) scores.masked_fill_(self.mask[:T, :T], float('-inf'))
        # 3) 缩放除以 sqrt(d_k)
        # 4) softmax -> dropout -> 乘 V
        ...


# 验证
ca = CausalAttention(d_in=3, d_out=4, context_length=6, dropout=0.0)
out = ca(xb)
print('输出 shape:', out.shape)

# 手动看权重:把 mask 后的 softmax 打印出来
with torch.no_grad():
    Q = ca.W_Q(xb); K = ca.W_K(xb)
    s = Q @ K.transpose(-2, -1) / (ca.d_out ** 0.5)
    s = s.masked_fill(ca.mask[:6, :6], float('-inf'))
    w = F.softmax(s, dim=-1)
    print('\ncausal weights (下三角形式):')
    print(w[0].round(decimals=3))

---
## 4. Multi-Head Attention(多头并行)

**思路**:把 d_out 维的 attention 拆成 h 份(每份 d_out/h 维),
每份独立算 attention,最后拼回去。

**好处**:不同的头可以关注不同的模式(语法、语义、位置距离...)。

**工程实现**:不用 for 循环写 h 遍 attention,直接 reshape 把头拆出来,
放到 batch 维旁边一起算。

```
x:     (B, T, d_in)
Q:     (B, T, d_out)
拆头:  (B, T, h, head_dim)   # d_out = h * head_dim
转置:  (B, h, T, head_dim)   # 把 h 放到 batch 旁边
算 attention,结果 (B, h, T, head_dim)
合并:  (B, T, h*head_dim) = (B, T, d_out)
最后再过一个输出投影 W_O
```

In [ ]:
# TODO: 实现 MultiHeadAttention

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, num_heads, dropout=0.0, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, 'd_out 必须能被 num_heads 整除'
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_Q = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_K = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_V = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_O = nn.Linear(d_out, d_out)                 # 合并后的输出投影
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1).bool()
        )

    def forward(self, x):
        B, T, _ = x.shape

        Q = self.W_Q(x)  # (B, T, d_out)
        K = self.W_K(x)
        V = self.W_V(x)

        # TODO: 把 d_out 拆成 (num_heads, head_dim),再 transpose 把头维放到 batch 旁边
        # Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)  # (B, h, T, hd)
        # K = K.view(...).transpose(1, 2)
        # V = V.view(...).transpose(1, 2)
        ...

        # TODO: 算缩放点积 attention(带 causal mask)
        # scores  = Q @ K.transpose(-2, -1) / sqrt(head_dim)       # (B, h, T, T)
        # scores  = scores.masked_fill(self.mask[:T, :T], float('-inf'))
        # weights = softmax(scores, -1) -> dropout
        # ctx     = weights @ V                                     # (B, h, T, hd)
        ...

        # TODO: 合并头:transpose 回来 + reshape 成 (B, T, d_out)
        # ctx = ctx.transpose(1, 2).contiguous().view(B, T, self.d_out)
        ...

        # TODO: 过输出投影 W_O 再返回
        ...


# 验证
mha = MultiHeadAttention(d_in=3, d_out=8, context_length=6, num_heads=2, dropout=0.0)
out = mha(xb)
print('输出 shape:', out.shape)  # (1, 6, 8)
n_params = sum(p.numel() for p in mha.parameters())
print('参数量:', n_params)

---
## 5. 封装到 `llm/attention.py`

In [ ]:
import pathlib

code = '''
import torch
import torch.nn as nn
import torch.nn.functional as F


class MultiHeadAttention(nn.Module):
    """带因果掩码的多头自注意力。"""

    def __init__(self, d_in, d_out, context_length, num_heads,
                 dropout=0.0, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_Q = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_K = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_V = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_O = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1).bool(),
        )

    def forward(self, x):
        B, T, _ = x.shape
        Q = self.W_Q(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.W_K(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.W_V(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1) / (self.head_dim ** 0.5)
        scores = scores.masked_fill(self.mask[:T, :T], float("-inf"))
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        ctx = weights @ V

        ctx = ctx.transpose(1, 2).contiguous().view(B, T, self.d_out)
        return self.W_O(ctx)
'''.lstrip()

out = pathlib.Path('../llm/attention.py')
out.write_text(code, encoding='utf-8')
print(f'已写入 {out.resolve()}')

# 验证
import sys, importlib
if 'llm.attention' in sys.modules:
    importlib.reload(sys.modules['llm.attention'])
from llm.attention import MultiHeadAttention as MHA
mha2 = MHA(d_in=3, d_out=8, context_length=6, num_heads=2)
_out = mha2(xb)
assert _out.shape == (1, 6, 8)
print('llm.attention 验证通过')

---
## 章末思考题

1. 为什么 attention 的 Q、K、V 要分三个独立的投影矩阵?如果让它们共享权重会怎样?
2. causal mask 在推理阶段(一次生成一个 token)还需要吗?为什么?
3. 多头 attention 的「多头」是怎么实现并行的?d_out=768、num_heads=12,每个头的维度是多少?

答完进入第 4 章 —— 把积木拼成完整 GPT。